## Install transformers library

In [1]:
%%capture
!pip install transformers
!pip install datasets
!pip install accelerate

### Check assigned GPU


In [2]:
!nvidia-smi

Tue Mar 12 08:54:53 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              11W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

# **Fine-tune Bert for Text classification**

## Fetch 20newsgroups dataset and split into train and dev sets


In [3]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

def read_20newsgroups(test_size=0.3):
  # download & load 20newsgroups dataset from sklearn's repos
  dataset = fetch_20newsgroups(subset="train", shuffle=True)
  documents = dataset.data
  labels = dataset.target
  # split into training & testing a return data as well as label names
  return train_test_split(documents, labels, test_size=test_size, random_state=12547392), dataset.target_names

# call the function
(train_texts, val_texts, train_labels, val_labels), target_names = read_20newsgroups()

### Data pre-processing/cleanning

In [4]:
def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

In [5]:
import spacy
import string
import numpy as np
from tqdm import tqdm

nlp = spacy.load('en_core_web_sm', disable=["tagger", "parser","ner", "lemmatizer"])
from spacy.lang.en.stop_words import STOP_WORDS
nlp.add_pipe('sentencizer')

X_train_tokenized = []
for idx in tqdm(range(len(train_texts))):
  doc = nlp(train_texts[idx])
  tokens = []
  for sent in doc.sents:
    for tok in sent:
      if '\n' in tok.text or "\t" in tok.text or "--" in tok.text or "*" in tok.text or\
       tok.text.lower() in STOP_WORDS or tok.text in string.punctuation or\
        all(x in string.punctuation for x in tok.text) or is_number(tok.text):
        continue
      if tok.text.strip():
        tokens.append(tok.text.replace('"',"'").strip().lower())
  X_train_tokenized.append(tokens)

X_val_tokenized = []
for idx in tqdm(range(len(val_texts))):
  doc = nlp(val_texts[idx])
  tokens = []
  for sent in doc.sents:
    for tok in sent:
      if '\n' in tok.text or "\t" in tok.text or "--" in tok.text or "*" in tok.text or\
       tok.text.lower() in STOP_WORDS or tok.text in string.punctuation or\
        all(x in string.punctuation for x in tok.text) or is_number(tok.text):
        continue
      if tok.text.strip():
        tokens.append(tok.text.replace('"',"'").strip().lower())
  X_val_tokenized.append(tokens)

100%|██████████| 3395/3395 [01:40<00:00, 33.90it/s]


In [6]:
X_train_clean= [" ".join(x) for x in X_train_tokenized]
X_valid_clean = [" ".join(x) for x in X_val_tokenized]

In [7]:
train_texts[0], X_train_clean[0], train_labels[0] , target_names

('From: kastle@wpi.WPI.EDU (Jacques W Brouillette)\nSubject: Re: WARNING.....(please read)...\nOrganization: Worcester Polytechnic Institute\nLines: 8\nDistribution: world\nNNTP-Posting-Host: wpi.wpi.edu\nKeywords: BRICK, TRUCK, DANGER\n\nCould we plase cease this discussion.  I fail to see why people feel the need \nto expound upon this issue for days and days on end.  These areas are not meant for this type of discussion.  If you feel the need to do such things, please\ntake your thought elsewhere.  Thanks.\n-- \n : I want only two things from this world, a 58 Plymouth and a small  : \n : OPEC nation with which to fuel it.  This would be a good and just  :\n : thing.  Car Smashers can just go home and sulk.                    :\n :        Jacques Brouillette ---  Manufacturing Engineering          :\n',
 'kastle@wpi wpi.edu jacques w brouillette subject warning (please read organization worcester polytechnic institute lines distribution world nntp posting host wpi.wpi.edu keywords br

In [8]:
from datasets.dataset_dict import DatasetDict
from datasets import Dataset

d = {'train':Dataset.from_dict({'label':train_labels,'text':X_train_clean}),
     'val':Dataset.from_dict({'label':val_labels,'text':X_valid_clean})
     }

news_dataset = DatasetDict(d)

In [9]:
news_dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 7919
    })
    val: Dataset({
        features: ['label', 'text'],
        num_rows: 3395
    })
})

In [10]:
news_dataset["train"][133]

{'label': 2,
 'text': 'mb4008@ehibm6.cen.uiuc.edu morgan j bullard subject hard drive compression ie stacker.superstor etc summary looking comparsions hard drive compression utilitys keywords stacker superstor doubledisk doublespace article i.d. news c5w8r9.ebu distribution comp.os.ms windows comp.os.ms-windows.apps organization university illinois urbana lines wondering knew hard drive compression utilities work hard drive getting want buy new intrested speed ease use compression aspect think important use things thanks morgan bullard mb4008@coewl.cen.uiuc.edu mjbb@uxa.cso.uiuc.edu'}

# Bert for text classification
 Pick a pretrained model from huggingface [hub](https://huggingface.co/models)

In [11]:
model_name = "bert-base-uncased"

### BPE Tokenizer

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [13]:
def bpe_tokenization(dataset):
    return tokenizer(dataset["text"], truncation=True)



### Apply BPE tokenizer to 20newsgroups dataset


In [14]:
tokenized_news_dataset = news_dataset.map(bpe_tokenization, batched=True)

Map:   0%|          | 0/7919 [00:00<?, ? examples/s]

Map:   0%|          | 0/3395 [00:00<?, ? examples/s]

In [15]:
print(tokenized_news_dataset["train"][11])
print(tokenizer.convert_ids_to_tokens(tokenized_news_dataset["train"][11]["input_ids"]))

{'label': 12, 'text': 'fculpepp@norfolk.vak12ed.edu fred w. culpepper subject cad program electronics organization virginia public education network norfolk lines making search cad program decent job making schematic drawings program needs ms dos windows possible want cad program draw diagrams dragging elements screen elements needed diverse vacuum tubes ics case pins needs provision adding legends components values words want produce quality drawings printout pin dot matrix and/or laser printer know cad program reasonable cost respond fred w. culpepper old dominion university retired fculpepp@norfolk.vak12ed.edu', 'input_ids': [101, 4429, 5313, 5051, 9397, 1030, 7735, 1012, 12436, 2243, 12521, 2098, 1012, 3968, 2226, 5965, 1059, 1012, 12731, 14277, 13699, 4842, 3395, 28353, 2565, 8139, 3029, 3448, 2270, 2495, 2897, 7735, 3210, 2437, 3945, 28353, 2565, 11519, 3105, 2437, 8040, 28433, 4588, 9254, 2565, 3791, 5796, 9998, 3645, 2825, 2215, 28353, 2565, 4009, 26309, 11920, 3787, 3898, 3787

### PAD sequences
Use **DataCollatorWithPadding** to create a batch of examples. It will also dynamically pad your text to the length of the longest element in its batch, so they are a uniform length. While it is possible to pad your text in the tokenizer function by setting padding=True, dynamic padding is more efficient.

In [16]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Load pretrained model

In [17]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(target_names)).to("cuda")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Define evaluation metrics

In [18]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)

  # calculate accuracy using sklearn's function
  acc = accuracy_score(labels, preds)
  f1 = f1_score(labels, preds,average='macro')

  return {
      'val_accuracy': acc,
      'val_f1': f1,
  }

### Define training arguments and fine-tune model
[TrainingArguments documentation](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/trainer#transformers.TrainingArguments)

In [19]:
training_args = TrainingArguments(
    output_dir='./results',           # output directory
    num_train_epochs=8,               # total number of training epochs
    per_device_train_batch_size=16,   # batch size per device during training
    per_device_eval_batch_size=16,    # batch size for evaluation
    gradient_accumulation_steps=2,    # Number of update steps (forward passes) to accumulate the gradients for, before performing a backward/update pass
    weight_decay=0.01,                # strength of weight decay
    logging_dir='./logs',             # directory for storing logs
    warmup_steps=500,                 # number of warmup steps for learning rate scheduler
    eval_steps=100,
    save_steps=100,
    evaluation_strategy="steps",
    metric_for_best_model = "val_f1",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,                                   # the instantiated Transformers model to be trained
    args=training_args,                            # training arguments, as defined above
    train_dataset=tokenized_news_dataset["train"], # training dataset
    eval_dataset=tokenized_news_dataset["val"],    # evaluation dataset
    tokenizer=tokenizer,                           # tokenizer
    data_collator=data_collator,                   # batch creator (padding)
    compute_metrics=compute_metrics                # the callback that computes metrics of interest
)

# Start training
trainer.train()

Step,Training Loss,Validation Loss,Val Accuracy,Val F1
100,No log,2.578233,0.406186,0.334012
200,No log,1.509465,0.655670,0.611551
300,No log,0.849529,0.802651,0.778340


OutOfMemoryError: CUDA out of memory. Tried to allocate 192.00 MiB. GPU 0 has a total capacty of 14.75 GiB of which 135.06 MiB is free. Process 15661 has 14.61 GiB memory in use. Of the allocated memory 11.17 GiB is allocated by PyTorch, and 3.31 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

### Performance of fine-tuned model on Dev set

In [ ]:
trainer.evaluate()

### Save the fine tuned model & tokenizer

In [ ]:
model_path = "20newsgroups-bert-base-uncased"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

### Reload fine-tunned model/tokenizer.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=len(target_names)).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
def get_prediction(text):
    # prepare our text into tokenized sequence
    inputs = tokenizer(text, truncation=True, return_tensors="pt").to("cuda")
    # perform inference to our model
    outputs = model(**inputs)
    # get output probabilities by doing softmax
    probs = outputs[0].softmax(1)
    # executing argmax function to get the candidate label
    return target_names[probs.argmax()]

In [ ]:
# Example #1
text = """
The first thing is first.
If you purchase a Macbook, you should not encounter performance issues that will prevent you from learning to code efficiently.
However, in the off chance that you have to deal with a slow computer, you will need to make some adjustments.
Having too many background apps running in the background is one of the most common causes.
The same can be said about a lack of drive storage.
For that, it helps if you uninstall xcode and other unnecessary applications, as well as temporary system junk like caches and old backups.
"""
print(get_prediction(text))

In [ ]:
# Example #2
text = """
A black hole is a place in space where gravity pulls so much that even light can not get out.
The gravity is so strong because matter has been squeezed into a tiny space. This can happen when a star is dying.
Because no light can get out, people can't see black holes.
They are invisible. Space telescopes with special tools can help find black holes.
The special tools can see how stars that are very close to black holes act differently than other stars.
"""
print(get_prediction(text))

In [ ]:
# Example #3
text = """
Coronavirus disease (COVID-19) is an infectious disease caused by a newly discovered coronavirus.
Most people infected with the COVID-19 virus will experience mild to moderate respiratory illness and recover without requiring special treatment.
Older people, and those with underlying medical problems like cardiovascular disease, diabetes, chronic respiratory disease, and cancer are more likely to develop serious illness.
"""
print(get_prediction(text))

## **Fine-tune Bert for Name Entity Recognition**


### Load WNUT 17 dataset

In [ ]:
model_name = "bert-base-uncased"

In [ ]:
from datasets import load_dataset
wnut = load_dataset("wnut_17")

In [ ]:
wnut

In [ ]:
# merge train & validation sets
from datasets import concatenate_datasets

train_dataset = concatenate_datasets([wnut["train"],wnut["validation"]])
train_dataset

The ner_tag describes an entity, such as a corporation, location, or person. The letter that prefixes each ner_tag indicates the token position of the entity:

- B- indicates the beginning of an entity.
- I- indicates a token is contained inside the same entity (e.g., the State token is a part of an entity like Empire State Building).
- 0 indicates the token doesn’t correspond to any entity.

In [ ]:
label_list = wnut["train"].features[f"ner_tags"].feature.names
# print tags set
label_list

### Preprocess

### Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

Since the input has already been split into words, set `is_split_into_words=True` to tokenize the words into subwords:

In [ ]:
tokenized_input = tokenizer(wnut["train"][0]["tokens"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
print(wnut["train"][0]["tokens"])
print(tokens)

Adding the special tokens [CLS] and [SEP] and subword tokenization creates a mismatch between the input and labels. A single word corresponding to a single label may be split into two subwords. You will need to realign the tokens and labels by:

1. Mapping all tokens to their corresponding word with the word_ids method.
2. Assigning the label -100 to the special tokens [CLS] and [SEP] so the loss function ignores them.
3. Only labeling the first token of a given word. Assign -100 to other subtokens from the same word.

Here is how you can create a function to realign the tokens and labels, and truncate sequences to be no longer than model's maximum input length:

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

Use Datasets map function to tokenize and align the labels over the entire dataset. You can speed up the map function by setting batched=True to process multiple elements of the dataset at once:

In [ ]:
tokenized_wnut = wnut.map(tokenize_and_align_labels, batched=True)
tokenized_train_dataset = train_dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
tokenized_train_dataset[1]

### Use DataCollator to dynamically pad sequences and make batches

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

### Load pretrained model

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(label_list)).to("cuda")

### Fine tune model

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    logging_dir='./logs',
    warmup_steps=500,
    eval_steps=20,
    save_steps=20,
    evaluation_strategy="steps",
    load_best_model_at_end=True,
    save_total_limit = 3
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_wnut["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

### Evaluate fine-tuned model on validation set


In [ ]:
trainer.evaluate()

In [ ]:
id2tag = {id: tag for id, tag in enumerate(label_list)}

In [ ]:
id2tag

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)


def get_prediction(text):
    # prepare our text into tokenized sequence
    inputs = tokenizer(text, truncation=True, return_tensors="pt").to("cuda")
    # perform inference to our model
    outputs = model(**inputs)
    # get output probabilities by doing softmax
    probs = outputs[0][0].softmax(1)
    # executing argmax function to get the candidate tags
    tokens_n_tags = [(tokenizer.decode(inputs['input_ids'][0][i].item()), id2tag[tag_id.item()])
                  for i, tag_id in enumerate (probs.argmax(axis=1))]

    return pd.DataFrame(tokens_n_tags, columns=['token', 'tag'])

In [ ]:
# Example #1
text1 = """
President Joe Biden is "fine" after tripping and falling over at an event in
Colorado, White House officials say."""

print(get_prediction(text1))

In [ ]:
# Example #2
text2 = """
Apple in October 2021 overhauled the high-end MacBook Pro, introducing
an entirely new design, new chips, new capabilities, and more."""

print(get_prediction(text2))

### Use already fine-tuned models for NER from Huggingface hub
https://huggingface.co/models

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

tokenizer_ner = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model_ner = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

nlp = pipeline("ner", model=model_ner, tokenizer=tokenizer_ner)

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)

def _get_predictions(text, tokenizer, model):

  text_tokens = tokenizer.convert_ids_to_tokens(tokenizer(text)["input_ids"])
  text_tags = ["O"]*len(text_tokens)
  pred_tags = model(text)
  for pr_tag in pred_tags:
    text_tags[pr_tag["index"]] = pr_tag["entity"]

  tokens_n_tags = [(word, tag) for word, tag in zip(text_tokens, text_tags)]

  return pd.DataFrame(tokens_n_tags, columns=['token', 'tag'])

In [ ]:
print(_get_predictions(text1, tokenizer_ner, nlp))

In [ ]:
print(_get_predictions(text2, tokenizer_ner, nlp))

# Resources
* https://huggingface.co/docs/transformers/tasks/sequence_classification
* https://huggingface.co/docs/transformers/tasks/token_classification